# Paper Table 1 evaluation

This notebook produces the per-stage evaluation table used in Sec. 9 of the ISMIR paper *"Raw Note Transcription for Hardanger Fiddle via a Hybrid Neural/Rule-Based Approach"*.

For each tune (Spretten, Godvaersdagen) and each post-processing stage (raw Transkun MIDI, +pitch refinement CSV, +offset synchronisation CSV) it compares against the performer-annotated ground truth (`*_truther*.csv`) and computes:

- **Standard F1** — onset tol 50 ms, offset tol max(50 ms, 20% dur), pitch tol 50 cents (MIREX/MAESTRO defaults: see `references.bib`).
- **Strict F1** — offset tol max(25 ms, 5% dur), same other tolerances. Sensitivity variant; not a separately named metric.
- **Deviation MAE** on matched notes — onset MAE [ms], offset MAE [ms], pitch MAE [cents].

Outputs:
- `table_results.csv` — full table (per-tune rows + aggregate).
- `table_results.tex` — LaTeX `tabular` ready to drop into the paper.

In [ ]:
from pathlib import Path
import sys, glob
import numpy as np
import pandas as pd

# eval_utils.py lives next to this notebook
sys.path.insert(0, str(Path.cwd()))
import eval_utils as ev

POSTPROS = Path('postpros')
TUNES = ['Spretten', 'Godvaersdagen']
STAGES = ['raw', '+pitch', '+offset']

In [ ]:
def resolve(tune):
    truth = sorted(glob.glob(str(POSTPROS / f'{tune}_truther*.csv')))[0]
    if tune == 'Spretten':
        raw = str(POSTPROS / 'Spretten_original_transkun239.mid')
        pitch = sorted(glob.glob(str(POSTPROS / 'Spretten_original_transkun239_pitch*.csv')))[0]
        offset = sorted(glob.glob(str(POSTPROS / 'Spretten_original_transkun239_offset*.csv')))[0]
    elif tune == 'Godvaersdagen':
        raw = str(POSTPROS / 'Godvaersdagen_original1_transkun239.mid')
        pitch = sorted(glob.glob(str(POSTPROS / 'Godvaersdagen_original1_transkun239_pitch*.csv')))[0]
        offset = sorted(glob.glob(str(POSTPROS / 'Godvaersdagen_original1_transkun239_offset*.csv')))[0]
    else:
        raise KeyError(tune)
    return {'truth': truth, 'raw': raw, '+pitch': pitch, '+offset': offset}

PATHS = {t: resolve(t) for t in TUNES}
for tune, p in PATHS.items():
    print(tune)
    for k, v in p.items():
        print(f'  {k:8s} {Path(v).name}')

In [ ]:
# Detect when +pitch and +offset CSVs are identical on (onset, offset, onpitch).
# This has happened when the offset-stage post-processor has not yet been re-exported.
def stages_identical(p1, p2):
    a = pd.read_csv(p1)[['onset', 'offset', 'onpitch']].dropna()
    b = pd.read_csv(p2)[['onset', 'offset', 'onpitch']].dropna()
    if a.shape != b.shape:
        return False
    return float((a.values - b.values).__abs__().sum()) < 1e-9

for tune in TUNES:
    same = stages_identical(PATHS[tune]['+pitch'], PATHS[tune]['+offset'])
    if same:
        print(f'WARNING: {tune}: +pitch and +offset CSVs have identical onset/offset/onpitch.')
        print(f'         The +offset row will repeat the +pitch row until the offset-stage')
        print(f'         post-processor is re-exported with actual offset corrections.')

In [ ]:
rows = []
for tune in TUNES:
    truth = PATHS[tune]['truth']
    for stage in STAGES:
        est = PATHS[tune][stage]
        r = ev.evaluate_pair(truth, est)
        rows.append({'tune': tune, 'stage': stage, **r})
        print(f"{tune:18s} {stage:8s} F_std={r['F_std']:.3f}  F_strict={r['F_strict']:.3f}  "
              f"onset_mae={r['onset_mae_ms']:5.2f}ms  offset_mae={r['offset_mae_ms']:5.2f}ms  "
              f"pitch_mae={r['pitch_mae_cents']:5.2f}ct  n_match={r['n_match_std']}/{r['n_match_offset']}")

per_pair = pd.DataFrame(rows)
per_pair

In [ ]:
# Note-weighted aggregate over the two tunes (MIREX micro-averaging convention)
def weighted_mean(df, value_col, weight_col='n_ref'):
    w = df[weight_col]
    return float((df[value_col] * w).sum() / w.sum())

agg_rows = []
for stage in STAGES:
    sub = per_pair[per_pair['stage'] == stage]
    agg_rows.append({
        'tune': 'AGGREGATE',
        'stage': stage,
        'n_ref': int(sub['n_ref'].sum()),
        'n_est': int(sub['n_est'].sum()),
        'P_std': weighted_mean(sub, 'P_std'),
        'R_std': weighted_mean(sub, 'R_std'),
        'F_std': weighted_mean(sub, 'F_std'),
        'P_strict': weighted_mean(sub, 'P_strict'),
        'R_strict': weighted_mean(sub, 'R_strict'),
        'F_strict': weighted_mean(sub, 'F_strict'),
        'onset_mae_ms': weighted_mean(sub, 'onset_mae_ms'),
        'offset_mae_ms': weighted_mean(sub, 'offset_mae_ms'),
        'pitch_mae_cents': weighted_mean(sub, 'pitch_mae_cents'),
        'n_match_std': int(sub['n_match_std'].sum()),
        'n_match_offset': int(sub['n_match_offset'].sum()),
    })

agg = pd.DataFrame(agg_rows)
agg

In [ ]:
# Sanity: strict F1 cannot exceed standard F1 (stricter offset constraint <=> fewer matches)
for _, row in per_pair.iterrows():
    assert row['F_strict'] <= row['F_std'] + 1e-9, \
        f"Strict > Standard at {row['tune']}/{row['stage']}"
print('All (tune, stage) rows: F_strict <= F_std OK')

In [ ]:
out_csv = Path('table_results.csv')
combined = pd.concat([per_pair, agg], ignore_index=True)
combined.to_csv(out_csv, index=False, float_format='%.4f')
print(f'Wrote {out_csv.resolve()}')

# LaTeX table for paper Sec 9 (aggregate row only - matches the paper's tabular spec)
stage_labels = {
    'raw': 'Raw model',
    '+pitch': '+ pitch refinement',
    '+offset': '+ offset synchronisation',
}

EOL = chr(92) + chr(92)  # two literal backslashes for end-of-row

def latex_row(stage_label, row):
    return (f"{stage_label} & {row['F_std']*100:.2f} & {row['F_strict']*100:.2f} & "
            f"{row['onset_mae_ms']:.1f} & {row['offset_mae_ms']:.1f} & {row['pitch_mae_cents']:.1f} {EOL}")

latex = []
latex.append(r'\begin{tabular}{lccccc}')
latex.append(r'\hline')
latex.append(f'Stage & F1 & F1 & Onset & Offset & Pitch {EOL}')
latex.append(f'      & (std) & (strict) & MAE [ms] & MAE [ms] & MAE [ct] {EOL}')
latex.append(r'\hline')
for stage in STAGES:
    row = agg[agg['stage'] == stage].iloc[0]
    latex.append(latex_row(stage_labels[stage], row))
latex.append(r'\hline')
latex.append(r'\end{tabular}')

out_tex = Path('table_results.tex')
out_tex.write_text('\n'.join(latex) + '\n')
print(f'Wrote {out_tex.resolve()}')
print()
print('\n'.join(latex))


## Caveats

1. **Raw stage pitch MAE is a quantization floor.** The raw Transkun output has integer MIDI pitches; the truth has fractional pitches (e.g. 72.14 = MIDI 72 + 14 cents). Pitch MAE on the raw stage is bounded below by the cents-deviation of truth from the nearest semitone. For Hardanger fiddle (non-equal temperament + scordatura) this floor is non-trivial. Part of any raw → +pitch change is dequantization, not pure model improvement.
2. **MAE is conditional on a match.** Read `n_match_std` (used for onset and pitch MAE) and `n_match_offset` (used for offset MAE) alongside the deviation values — a stage with lower MAE but a lower match count is not necessarily better.
3. **+offset stage data caveat (2026-04-27).** The postpros `*_offset.csv` files currently differ from `*_pitch.csv` only in alignment metadata (`previous`, `next` columns), not in onset/offset/pitch. The +offset row therefore repeats the +pitch row until the offset-stage post-processor is re-exported with actual offset corrections. The check in the third code cell prints a warning when this is the case.